# First PyTorch Model: A Small LSTM

This notebook introduces the first neural network in the project. The point is not tuning. The point is to understand how tensors move through a minimal LSTM training pipeline.

## Terms

- **Tensor**: a multi-dimensional array. `X` is shaped `(examples, 60, 28)` and `y` is shaped `(examples, 7)`.
- **Parameter**: a learnable number inside the model.
- **Batch**: a small group of examples processed at once.
- **Dataset**: returns one `(X, y)` pair by index.
- **DataLoader**: creates batches from a Dataset.
- **Forward pass**: sends input through the model to get predictions.
- **Loss**: measures prediction error. We use MSE.
- **Gradient**: tells each parameter how to move to reduce loss.
- **Backpropagation**: computes gradients through the model.
- **Optimizer**: updates parameters. We use Adam.
- **Epoch**: one full pass through the training data.

In [ ]:
# ruff: noqa: E402, I001
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from market_resonance.features import (
    build_supervised_windows,
    chronological_train_validation_test_split,
    standardize_splits,
)
from market_resonance.models import YieldCurveLSTM
from market_resonance.training import (
    TrainingConfig,
    make_dataloaders,
    set_deterministic_seed,
    train_lstm_model,
)

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "treasury_yields_daily.csv"
CHECKPOINT_PATH = PROJECT_ROOT / "reports" / "models" / "first_lstm.pt"

## Build tensors and loaders

We reuse the supervised windows, chronological split, and train-only normalization from the previous step.

In [ ]:
set_deterministic_seed(42)
yields = pd.read_csv(DATA_PATH, parse_dates=["date"])
windows = build_supervised_windows(yields, lookback=60, horizon=1)
splits = chronological_train_validation_test_split(windows)
standardized_splits, standardizer = standardize_splits(splits)
train_loader, validation_loader, test_loader = make_dataloaders(
    standardized_splits,
    batch_size=64,
    seed=42,
)
{
    "train_X": standardized_splits.train.X.shape,
    "validation_X": standardized_splits.validation.X.shape,
    "test_X": standardized_splits.test.X.shape,
}

## Shape trace through the model

The LSTM receives the whole 60-day sequence. It returns a hidden vector for every day. We keep the final day's hidden vector and pass it through a linear layer to get seven predictions.

In [ ]:
model = YieldCurveLSTM(input_size=28, hidden_size=64, output_size=7)
X_batch, y_batch = next(iter(train_loader))
model.shape_trace(X_batch)

## Minimal training run

This uses MSE loss, Adam, validation loss, early stopping, and a best-model checkpoint. The settings are intentionally plain.

In [ ]:
history = train_lstm_model(
    model=model,
    train_loader=train_loader,
    validation_loader=validation_loader,
    config=TrainingConfig(
        batch_size=64,
        learning_rate=1e-3,
        max_epochs=20,
        patience=3,
        seed=42,
    ),
    checkpoint_path=CHECKPOINT_PATH,
)
{
    "epochs_run": len(history.train_loss),
    "best_epoch": history.best_epoch,
    "best_validation_mse": history.best_validation_loss,
    "checkpoint": str(CHECKPOINT_PATH),
}

In [ ]:
pd.DataFrame(
    {
        "epoch": range(1, len(history.train_loss) + 1),
        "train_loss": history.train_loss,
        "validation_loss": history.validation_loss,
    }
)